# 02 - Model Extraction and Membership Inference

Two privacy attacks that need nothing but the prediction API:

- **Extraction** - query the target enough times to train a *surrogate* that
  mirrors its decisions. Once you own a high-fidelity copy, you attack it offline
  for free, or you have simply stolen the model.
- **Membership inference** - decide whether one specific record was in the
  training set. For a model trained on patients or customers, a confident "yes"
  is a privacy breach in a single query.

**Why it matters (CIA).** Both are **Confidentiality** attacks: extraction steals
the *model* (your training investment and IP), membership inference leaks *who was
in the training data* (a direct privacy/compliance breach). A high-fidelity clone
also lets an attacker craft evasions offline, so extraction feeds **Integrity**
attacks too.

Both run against the published **`ml-extraction-fraud-tabular`** environment, so
there is nothing to deploy.

> **New here? Run [`00_prerequisites.ipynb`](../00_prerequisites.ipynb) first** -
> install the CLI (`curl -fsSL https://dreadnode.io/install.sh | bash`), sign in
> (`dn login`), and create a workspace. Everything below streams findings to your
> Dreadnode workspace and draws from your credit balance.

> **Follow along in the docs:** [Extraction](https://docs.dreadnode.io/ai-red-teaming/learning-guide/extraction) and [Membership Inference](https://docs.dreadnode.io/ai-red-teaming/learning-guide/membership-inference) cover the concepts, threat models, and defenses in depth.

## Setup

## What we are assessing

We are going to assess two confidentiality risks to a classifier that only exposes
a `/predict` API: **model extraction** - can an attacker query it enough to train a
high-fidelity *clone* (steal the model) - and **membership inference** - can they
tell whether a specific record was in the training set (a privacy leak). We steal
the credit-card fraud model four ways and measure each clone's fidelity, then run
two membership attacks and measure their AUC.

Same rhythm as the other demos: first the target and the query data the attacker
gets, then each attack, then its metrics.

In [ ]:
# Traditional-ML notebooks train scikit-learn / torch surrogate models.
# If this fails, install the extra:  pip install "dreadnode[airt-ml]"
try:
    import sklearn  # noqa: F401
    import torch  # noqa: F401
except ModuleNotFoundError as exc:
    raise SystemExit(
        f"Missing '{exc.name}'. The traditional-ML notebooks need the airt-ml extra:\n"
        '  pip install "dreadnode[airt-ml]"'
    ) from exc

PROJECT = "airt-learning-02-extraction-membership"
ORG = "your-org-slug"  # your workspace slug from the platform URL
WORKSPACE = "main"

In [ ]:
import dreadnode as dn

instance = dn.configure(project=PROJECT, workspace=WORKSPACE, organization=ORG, console=False)
api = instance.api
print("configured; findings stream to project:", PROJECT)

In [ ]:
from dreadnode.airt import (
    PredictionTargetSpec,
    copycat_extraction,
    knockoff_extraction,
    shadow_model_membership,
    threshold_membership,
)
from dreadnode.airt.assessment import Assessment

In [ ]:
import asyncio
import httpx

from dreadnode.core.environment import TaskEnvironment


_ENVS: list[TaskEnvironment] = []


async def provision(task_ref: str, timeout: int = 180) -> tuple[TaskEnvironment, str]:
    """Spin up a published Dreadnode environment and return (env, base_url) once
    the classifier service is actually answering. `setup()` returns before the app
    binds its port, so we poll /pool until it responds."""
    env = TaskEnvironment(api, org=ORG, workspace=WORKSPACE, task_ref=task_ref)
    _ENVS.append(env)
    ctx = await env.setup()
    url = ctx["service_urls"]["challenge"]["url"]
    for _ in range(timeout // 5):
        try:
            if httpx.get(f"{url}/pool?n=1", timeout=15).status_code == 200:
                print(f"{task_ref} ready at {url}")
                return env, url
        except httpx.HTTPError:
            pass
        await asyncio.sleep(5)
    raise RuntimeError(f"{task_ref} did not become ready in {timeout}s")


def make_spec(url: str, num_classes: int, modality: str, name: str) -> PredictionTargetSpec:
    """Point a PredictionTargetSpec at the environment's /predict endpoint. Text
    targets take {"text": ...}; tabular/image targets take {"features": [...]}."""
    template = '{"text": "{input}"}' if modality == "text" else '{"features": {input}}'
    return PredictionTargetSpec(
        endpoint=f"{url}/predict",
        request_template=template,
        probabilities_path="$.probabilities",
        input_format="text" if modality == "text" else "json_array",
        num_classes=num_classes,
        name=name,
    )

## Provision the target once

We reuse the same environment for every attack in this notebook, tearing it down
at the end.

In [ ]:
env, url = await provision("ml-extraction-fraud-tabular")
spec = make_spec(url, num_classes=2, modality="tabular", name="Credit-card fraud (tabular)")

# /pool gives unlabeled query inputs; /members and /nonmembers give the labeled
# records the membership attacks need.
pool = httpx.get(f"{url}/pool?n=800", timeout=60).json()["inputs"]
members = httpx.get(f"{url}/members?n=200", timeout=60).json()
nonmembers = httpx.get(f"{url}/nonmembers?n=200", timeout=60).json()
print(f"pool={len(pool)}  members={len(members['records'])}  nonmembers={len(nonmembers['records'])}")

## Extraction - Knockoff Nets (soft labels)

**Knockoff** queries the target with the pool, records the full probability
vector for each input, and trains a surrogate on those soft labels. Soft labels
carry more signal than hard labels, so fidelity climbs fast. **Fidelity** is the
fraction of held-out inputs where the surrogate agrees with the target - the
closer to 1.0, the more completely the model was stolen.

We also pass a labeled held-out set (`eval_pool` + `ground_truth`) so the run
reports how the clone compares to the original in real terms:

- **KL divergence** - how close the clone's confidence is to the target's, not
  just its labels. Near zero means even the probability calibration was cloned.
- **Victim accuracy** and **accuracy retained** - the clone's accuracy as a
  fraction of the original's. Near 100% means the copy is as useful as the real
  model.

**Algorithm:** Knockoff Nets -
[Orekondy, Schiele & Fritz, 2018](https://arxiv.org/abs/1812.02766).

In [ ]:
# Non-members are labeled data the target was NOT trained on, so they make a fair
# held-out set for measuring how accurate the stolen clone really is.
async with Assessment("knockoff_extraction - fraud - dreadnode-env"):
    result = await knockoff_extraction(
        spec, query_pool=pool, eval_pool=nonmembers["records"],
        ground_truth=nonmembers["labels"], query_budget=600, num_classes=2,
        modality="tabular", measure_transfer=False, seed=0,
        airt_target_model="Credit-card fraud (tabular)",
    ).run()
md = result.metrics_detail
print(f"knockoff fidelity={result.fidelity:.3f}  queries={result.query_count}")
print(f"  KL divergence={md['kl_divergence']:.3f}  "
      f"victim acc={md.get('target_accuracy')}  retained={md.get('accuracy_ratio')}")

## Extraction - Copycat (hard labels)

**Copycat** is the weaker-assumption baseline: it trains only on the target's
top-1 label, the minimum any classifier must reveal. Comparing its fidelity to
Knockoff shows how much extra leverage those confidence scores hand an attacker.

**Algorithm:** Copycat CNN -
[Correia-Silva et al., 2018](https://arxiv.org/abs/1806.05476).

In [ ]:
async with Assessment("copycat_extraction - fraud - dreadnode-env"):
    result = await copycat_extraction(
        spec, query_pool=pool, query_budget=600, num_classes=2, modality="tabular",
        measure_transfer=False, seed=0, airt_target_model="Credit-card fraud (tabular)",
    ).run()
print(f"copycat fidelity={result.fidelity:.3f}  queries={result.query_count}")

## Membership inference - confidence threshold (Yeom 2018)

The simplest membership signal: models are more confident on data they were
trained on. This attack scores members vs. non-members by confidence and reports
**AUC** - 0.5 is a coin flip (private), well above 0.5 means the model leaks who
it was trained on.

**Algorithm:** Confidence-threshold membership inference -
[Yeom, Giacomelli, Fredrikson & Jha, IEEE CSF 2018](https://arxiv.org/abs/1709.01604).

In [ ]:
async with Assessment("threshold_membership - fraud - dreadnode-env"):
    result = await threshold_membership(
        spec, members=members["records"], nonmembers=nonmembers["records"],
        member_labels=members["labels"], nonmember_labels=nonmembers["labels"],
        num_classes=2, modality="tabular", seed=0,
        airt_target_model="Credit-card fraud (tabular)",
    ).run()
print(f"threshold AUC={result.auc:.3f}")

## Membership inference - shadow models (Shokri 2017)

The stronger attack trains **shadow models** that imitate the target, then trains
an attack classifier on their in/out behavior. It usually beats the simple
threshold because it learns the target's confidence signature instead of assuming
one.

**Algorithm:** Shadow-model membership inference -
[Shokri, Stronati, Song & Shmatikov, IEEE S&P 2017](https://arxiv.org/abs/1610.05820).

In [ ]:
async with Assessment("shadow_model_membership - fraud - dreadnode-env"):
    result = await shadow_model_membership(
        spec, members=members["records"], nonmembers=nonmembers["records"],
        member_labels=members["labels"], nonmember_labels=nonmembers["labels"],
        num_classes=2, modality="tabular", seed=0,
        airt_target_model="Credit-card fraud (tabular)",
    ).run()
print(f"shadow-model AUC={result.auc:.3f}")
await env.teardown()

## Read the findings

Open **[https://app.dreadnode.io](https://app.dreadnode.io)** -> your workspace -> project
**airt-learning-02-extraction-membership**. Extraction findings show surrogate
fidelity, KL divergence, and the clone's accuracy against the victim's; membership
findings show AUC, attack accuracy/precision/recall, TPR at low FPR, and a table of
re-identified records. Higher fidelity and higher AUC both mean "more exposed."

## Homework

- **Soft vs hard labels:** Knockoff (soft-label) beat Copycat (hard-label) on
  fidelity - by how much here? If you could only read the top-1 label, what would you
  do to close the gap?
- **Query budget vs fidelity:** halve `query_budget` and re-run extraction. Where
  does fidelity fall off? That curve is the target's real exposure to a rate-limited
  attacker.
- **Membership signal:** the threshold attack uses confidence; the shadow-model
  attack learns the signature. Which records are re-identified by both, and what do
  they have in common (outliers? duplicates?)?

## Clean up

Hosted environments keep billing compute until you release them. Tear down everything this notebook provisioned:

In [ ]:
for _e in _ENVS:
    await _e.teardown()
print(f"tore down {len(_ENVS)} environment(s)")

## Run it without a notebook (TUI)

Everything here is also driveable from the terminal - same platform, same findings:

- **TUI:** run `dreadnode` (no arguments) for the interactive terminal UI, pick the
  target and attack, and watch progress live.
